# Lab 18-07 · Scheduling, Blocking and Priority Inversion

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import rtsched as rs

## Part 1 · The two-task bound

*Predict* the smallest utilization of a just-schedulable two-task RM set, and the period ratio where it occurs.

In [2]:
u, ratio = rs.two_task_min_utilization(1000)
print(f"minimum U = {u:.5f} at T2/T1 = {ratio:.3f};  2(sqrt2 - 1) = {2 * (2 ** 0.5 - 1):.5f}")
f = np.linspace(0.01, 0.99, 200)
plt.plot(1 + f, (1 + f ** 2) / (1 + f)); plt.axhline(rs.ll_bound(2), color="grey", linestyle=":")
plt.xlabel("T2 / T1"); plt.ylabel("least utilization that just fails to fit"); plt.show()

minimum U = 0.82843 at T2/T1 = 1.414;  2(sqrt2 - 1) = 0.82843


## Part 2 · Bounds against exactness

*Predict* the fraction of random three-task sets each test accepts at U = 0.80 and 0.90.

In [3]:
for U in (0.80, 0.90):
    ll, hy, ex = rs.acceptance(rs.random_task_sets(2000, 3, U))
    print(f"U = {U:.2f}: Liu & Layland {ll:.3f}, hyperbolic {hy:.3f}, exact {ex:.3f}")

U = 0.80: Liu & Layland 0.000, hyperbolic 0.716, exact 1.000
U = 0.90: Liu & Layland 0.000, hyperbolic 0.055, exact 0.749


## Part 3 · RM against EDF

*Predict* when task B of Example 18-07.3 first misses under RM.

In [4]:
def ex3(cb=4):
    return rs.rm_priorities([rs.Task("A", 5, [(2, None)]), rs.Task("B", 7, [(cb, None)])])
for policy in ("fp", "edf"):
    r = rs.simulate(ex3(), 350, policy=policy)
    print(policy, "misses", r.misses, "worst", {k: r.worst(k) for k in r.response}, "first B responses", r.response["B"][:3])
print("first 35 ms under RM: ", "".join(x or "." for x in rs.simulate(ex3(), 35).trace))
print("first 35 ms under EDF:", "".join(x or "." for x in rs.simulate(ex3(), 35, policy="edf").trace))
for policy in ("fp", "edf"):
    print("overload, C_B = 5,", policy, rs.simulate(ex3(5), 350, policy=policy).misses)

fp misses {'A': 0, 'B': 10} worst {'A': 2, 'B': 8} first B responses [8, 7, 6]
edf misses {'A': 0, 'B': 0} worst {'A': 4, 'B': 6} first B responses [6, 5, 6]
first 35 ms under RM:  AABBBAABBBAABBBAABBBAABBBAABBBAABB.
first 35 ms under EDF: AABBBBAABBBBAABAABBBAABBBBAABBAABB.
overload, C_B = 5, fp {'A': 0, 'B': 49}
overload, C_B = 5, edf {'A': 64, 'B': 47}


## Part 4 · Priority inversion

*Predict* H's response for medium-task lengths of 1, 5 and 20 ms under each protocol.

In [5]:
def ex4(m):
    return [rs.Task("L", 100, [(1, None), (4, "S"), (1, None)], priority=1, offset=0, max_jobs=1),
            rs.Task("M", 100, [(m, None)], priority=2, offset=3, max_jobs=1),
            rs.Task("H", 100, [(1, None), (1, "S"), (1, None)], priority=3, offset=2, max_jobs=1)]
ms = list(range(1, 21))
for proto in ("none", "inherit", "ceiling"):
    resp = [rs.simulate(ex4(m), 80, protocol=proto).worst("H") for m in ms]
    plt.plot(ms, resp, "o-", label=proto)
    print(proto, [resp[m - 1] for m in (1, 5, 20)])
plt.xlabel("medium task length m (ms)"); plt.ylabel("H's response (ms)"); plt.legend(); plt.show()
for proto in ("none", "inherit", "ceiling"):
    print(f"{proto:8s}", "".join(x or "." for x in rs.simulate(ex4(10), 20, protocol=proto).trace))

none [7, 11, 26]
inherit [6, 6, 6]
ceiling [6, 6, 6]
none     LLHMMMMMMMMMMLLLHHL.
inherit  LLHLLLHHMMMMMMMMMML.
ceiling  LLLLLHHHMMMMMMMMMML.


## Part 5 · Queues

*Predict* the deepest queue for each stall.

In [6]:
for stall in ((30, 45), (31, 45), (30, 50)):
    print("stall", stall, "->", rs.queue_max_occupancy(10, 200, [stall]), "items")

stall (30, 45) -> 5 items
stall (31, 45) -> 4 items
stall (30, 50) -> 6 items


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).